# 5. Feature engineering

## 5.1. Feature engineering de las regiones

**1. Area de los municipios**



In [36]:
# Convertir el diccionario a GeoDataFrame
# Usualmente la información espacial está bajo la llave 'features'
gdf_dane = gpd.GeoDataFrame.from_features(geojson_dane['features'])


if gdf_dane.crs is None:
    gdf_dane.set_crs('EPSG:4326', inplace=True)

#Aplicar el cambio de proyección a MAGNA-SIRGAS
muns_shp = gdf_dane.to_crs('EPSG:9377')

muns_shp['area_km2'] = muns_shp.geometry.area / 1e6
muns_shp['cod_mun']  = muns_shp['mpio_cdpmp'].astype(str).str.zfill(5)  # ajusta nombre columna
area_df = muns_shp[['cod_mun','area_km2']]
print(f"✓ Área calculada desde shapefile: {len(area_df)} municipios")

✓ Área calculada desde shapefile: 1122 municipios


In [37]:
area_df

,cod_mun,area_km2
0,05001,374.741484
1,05002,506.959782
2,05004,296.974188
3,05021,128.856431
4,05030,84.117145
...,...,...
1117,97889,4640.894609
1118,99001,12142.323366
1119,99524,18492.107244
1120,99624,3679.005021


**2. Topografía  y altitud**

In [38]:
# Obtener coordenadas únicas por municipio
df_geo_uniq = df_modelo.groupby('cod_mun').agg(
    latitud =('latitud','first'),
    longitud=('longitud','first')
).dropna().reset_index()

print(f"Obteniendo altitud para {len(df_geo_uniq)} municipios...")


coords_list = list(zip(df_geo_uniq['latitud'], df_geo_uniq['longitud']))
altitudes   = obtener_altitud_lote(coords_list, batch_size=100)

df_geo_uniq['altitud_m'] = altitudes
df_geo_uniq['altitud_m'] = pd.to_numeric(df_geo_uniq['altitud_m'], errors='coerce')

df_geo_uniq['piso_termico']  = df_geo_uniq['altitud_m'].apply(clasificar_piso_termico)
df_geo_uniq['log_altitud']   = np.log1p(df_geo_uniq['altitud_m'].clip(lower=0))

print(f"\n✓ Altitud obtenida para {df_geo_uniq['altitud_m'].notna().sum()} municipios")
print(f"\nDistribución de pisos térmicos:")
print(df_geo_uniq['piso_termico'].value_counts().sort_index()
      .rename({0:'Caliente (0-900m)',1:'Templada (900-2000m)',2:'Fría (>2000m)'}))
print(f"\nAltitud (metros):")
print(df_geo_uniq['altitud_m'].describe().round(0))

FEATURES_TOPO = ['altitud_m', 'log_altitud', 'piso_termico']

Obteniendo altitud para 1091 municipios...
  Procesados 500/1091 municipios...
  Procesados 1000/1091 municipios...

✓ Altitud obtenida para 1091 municipios

Distribución de pisos térmicos:
piso_termico
Caliente (0-900m)       437
Templada (900-2000m)    369
Fría (>2000m)           285
Name: count, dtype: int64

Altitud (metros):
count    1091.0
mean     1273.0
std      1009.0
min         0.0
25%       232.0
50%      1272.0
75%      2050.0
max      3980.0
Name: altitud_m, dtype: float64


**3. Contexto multiescalar**

In [39]:
# ────────────────────────────────────────────────────────────
# Escala 1: municipio
# Escala 2: departamento
# Escala 3: región/macro
# ────────────────────────────────────────────────────────────

# Regiones macro de Colombia (5 regiones DANE)
REGIONES = {
    'Caribe':    ['08','13','20','23','44','47','70'],
    'Andina':    ['05','15','17','25','41','50','63','66','68','73','76','11'],
    'Pacífica':  ['19','27','52','76'],
    'Orinoquía': ['50','81','85','86'],
    'Amazonía':  ['18','86','91','94','95','97','99'],
}
dpto_region = {}
for region, dptos in REGIONES.items():
    for d in dptos:
        dpto_region[d] = region

df_base_ext = df_modelo.copy()
df_base_ext['cod_dpto'] = df_base_ext['cod_mun'].str[:2]
df_base_ext['region']   = df_base_ext['cod_dpto'].map(dpto_region).fillna('Otra')




**4. Interacciones**

In [40]:
# ────────────────────────────────────────────────────────────
# Capturar que el mismo nivel de pobreza tiene distinto
# efecto según el contexto geográfico-climático
# ────────────────────────────────────────────────────────────

# Top 3 indicadores IPM más correlacionados (del EDA)
TOP3_IPM = ['barreras_primera_infancia', 'tasa_dependencia', 'sin_agua_mejorada']

# Interacción: privación × altitud (el frío agrava la desnutrición)
# Interacción: privación × aridez (escasez de agua agrava la situación)
# Interacción: privación × ruralidad

for col_ipm in TOP3_IPM:
    if col_ipm in df_base_ext.columns:
        # × ruralidad (proxy: sin_agua_mejorada o el prop_rural)
        col_corta = col_ipm.replace('barreras_','').replace('_primera_infancia','_bi')
        df_base_ext[f'inter_{col_corta}_rural'] = (
            df_base_ext[col_ipm] * df_base_ext.get('prop_rurall', 0.5)
        )

FEATURES_INTERACC = [c for c in df_base_ext.columns if c.startswith('inter_')]
print(f"\n✓ Interacciones creadas: {FEATURES_INTERACC}")



✓ Interacciones creadas: ['inter_primera_infancia_rural', 'inter_tasa_dependencia_rural', 'inter_sin_agua_mejorada_rural']


In [41]:
df_base_ext

,cod_mun,año,NOMBRE,analfabetismo,bajo_logro_educativo,barreras_primera_infancia,barreras_acceso_salud,tasa_dependencia,hacinamiento_critico,inadec_eliminacion_excretas,...,Año,Componente de gestión,Componente de resultados,MDM,n_missings,cod_dpto,region,inter_primera_infancia_rural,inter_tasa_dependencia_rural,inter_sin_agua_mejorada_rural
0,05001,2018,Medellín,5.0,35.2,1.4,2.6,22.8,5.4,2.0,...,2018.0,81.30,72.56,80.58,0,05,Andina,0.7,11.40,0.75
1,05001,2019,Medellín,5.0,35.2,1.4,2.6,22.8,5.4,2.0,...,2019.0,85.29,69.14,83.60,0,05,Andina,0.7,11.40,0.75
2,05001,2020,Medellín,5.0,35.2,1.4,2.6,22.8,5.4,2.0,...,2020.0,80.51,72.92,82.32,0,05,Andina,0.7,11.40,0.75
3,05001,2021,Medellín,5.0,35.2,1.4,2.6,22.8,5.4,2.0,...,2021.0,82.73,74.92,83.69,0,05,Andina,0.7,11.40,0.75
4,05001,2022,Medellín,5.0,35.2,1.4,2.6,22.8,5.4,2.0,...,2022.0,83.36,74.56,83.19,0,05,Andina,0.7,11.40,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6396,99773,2020,Cumaribo,26.0,85.4,18.8,5.4,83.1,38.9,85.0,...,2020.0,20.32,43.13,20.26,0,99,Amazonía,9.4,41.55,39.55
6397,99773,2021,Cumaribo,26.0,85.4,18.8,5.4,83.1,38.9,85.0,...,2021.0,39.67,43.92,39.89,0,99,Amazonía,9.4,41.55,39.55
6398,99773,2022,Cumaribo,26.0,85.4,18.8,5.4,83.1,38.9,85.0,...,2022.0,44.40,49.69,46.18,0,99,Amazonía,9.4,41.55,39.55
6399,99773,2023,Cumaribo,26.0,85.4,18.8,5.4,83.1,38.9,85.0,...,2023.0,48.87,51.41,49.43,0,99,Amazonía,9.4,41.55,39.55


**5. Agregarlos a la BD**

In [42]:
# Base: df_multi (ya tiene multiescala + interacciones)
df_final = df_base_ext.copy()

# 1. Topografía + área (una fila por municipio → merge por cod_mun)

df_final = df_final.merge(
     area_df,
    on='cod_mun', how='left'
)

df_final = df_final.merge(
    df_geo_uniq[['cod_mun'] + FEATURES_TOPO],
    on='cod_mun', how='left'
)

**Distancia a capitales departamentales**

In [43]:
# Coordenadas de las 32 capitales departamentales de Colombia
CAPITALES = {
    '05': ('Medellín',        6.2476,  -75.5658),
    '08': ('Barranquilla',    10.9639, -74.7964),
    '11': ('Bogotá',           4.7110,  -74.0721),
    '13': ('Cartagena',        10.3910, -75.4794),
    '15': ('Tunja',             5.5353,  -73.3677),
    '17': ('Manizales',         5.0703,  -75.5138),
    '18': ('Florencia',         1.6144,  -75.6063),
    '19': ('Popayán',           2.4448,  -76.6147),
    '20': ('Valledupar',       10.4631,  -73.2532),
    '23': ('Montería',          8.7575,  -75.8908),
    '25': ('Cundinamarca',      4.7110,  -74.0721),  # Bogotá como proxy
    '27': ('Quibdó',            5.6947,  -76.6611),
    '41': ('Neiva',             2.9273,  -75.2819),
    '44': ('Riohacha',         11.5444,  -72.9072),
    '47': ('Santa Marta',      11.2408,  -74.1990),
    '50': ('Villavicencio',     4.1420,  -73.6266),
    '52': ('Pasto',             1.2136,  -77.2811),
    '54': ('Cúcuta',            7.8939,  -72.5078),
    '63': ('Armenia',           4.5339,  -75.6811),
    '66': ('Pereira',           4.8133,  -75.6961),
    '68': ('Bucaramanga',       7.1198,  -73.1227),
    '70': ('Sincelejo',         9.3047,  -75.3978),
    '73': ('Ibagué',            4.4389,  -75.2322),
    '76': ('Cali',              3.4516,  -76.5320),
    '81': ('Arauca',            7.0847,  -70.7581),
    '85': ('Yopal',             5.3378,  -72.3959),
    '86': ('Mocoa',             1.1521,  -76.6478),
    '88': ('San Andrés',       12.5567,  -81.7185),
    '91': ('Leticia',          -4.2153,  -69.9406),
    '94': ('Inírida',           3.8653,  -67.9239),
    '95': ('San José Guaviare', 2.5664,  -72.6416),
    '97': ('Mitú',              1.1983,  -70.1730),
    '99': ('Puerto Carreño',    6.1891,  -67.4833),
}


# Calcular distancias (una vez por municipio, luego merge)
df_geo = (
    df_modelo.groupby('cod_mun')
    .agg(latitud=('latitud','first'), longitud=('longitud','first'))
    .reset_index()
    .dropna()
)

print("Calculando distancias...")
df_geo['dist_capital_dpto']  = df_geo.apply(distancia_a_capital,              axis=1)
df_geo['dist_bogota']        = df_geo.apply(distancia_a_bogota,               axis=1)
df_geo['dist_ciudad_cercana']= df_geo.apply(distancia_a_ciudad_mas_cercana,   axis=1)

# Unir al panel
df_final = df_final.merge(
    df_geo[['cod_mun','dist_capital_dpto','dist_bogota','dist_ciudad_cercana']],
    on='cod_mun', how='left'
)

FEATURES_GEO = ['dist_capital_dpto', 'dist_bogota', 'dist_ciudad_cercana']

print("\n=== DISTANCIAS A CENTROS URBANOS (km) ===")
print(df_final[FEATURES_GEO].describe().round(1))
print("\nInterpretación esperada: municipios más alejados → mayor riesgo")

# Verificar correlación con tasa
for col in FEATURES_GEO:
    r = df_final[['tasa_x1000', col]].corr().iloc[0,1]
    print(f"  Correlación {col} vs tasa: {r:.3f}")



Calculando distancias...

=== DISTANCIAS A CENTROS URBANOS (km) ===
       dist_capital_dpto  dist_bogota  dist_ciudad_cercana
count             6324.0       6324.0               6324.0
mean                78.3        316.7                 62.6
std                 59.9        190.7                 39.0
min                  1.6         10.2                  1.6
25%                 37.1        164.3                 35.4
50%                 63.0        284.0                 55.0
75%                 99.4        467.1                 78.9
max                381.7       1251.3                267.5

Interpretación esperada: municipios más alejados → mayor riesgo
  Correlación dist_capital_dpto vs tasa: 0.062
  Correlación dist_bogota vs tasa: -0.046
  Correlación dist_ciudad_cercana vs tasa: 0.092


**Tasa del año anterior por municipio**

**1. Contruir población para 2016 - 2017**

In [44]:
# Verificar columnas de edad disponibles
cols_edad_total = [c for c in pob_raw_17.columns if c.startswith('Total_')]
print(f"Columnas edad disponibles: {cols_edad_total[:10]}...")

# Columnas 0-4 años en este formato
COLS_0_4_17 = ['Total_0', 'Total_1', 'Total_2', 'Total_3', 'Total_4']

# Verificar que existen
cols_disponibles = [c for c in COLS_0_4_17 if c in pob_raw_17.columns]
cols_faltantes   = [c for c in COLS_0_4_17 if c not in pob_raw_17.columns]
print(f"\nCols 0-4 encontradas: {cols_disponibles}")
if cols_faltantes:
    print(f"⚠ No encontradas: {cols_faltantes}")

Columnas edad disponibles: ['Total_0', 'Total_1', 'Total_2', 'Total_3', 'Total_4', 'Total_5', 'Total_6', 'Total_7', 'Total_8', 'Total_9']...

Cols 0-4 encontradas: ['Total_0', 'Total_1', 'Total_2', 'Total_3', 'Total_4']


In [45]:
# ── Filtrar solo área Total y calcular pob_0_4 ──
pob_17 = pob_raw_17[pob_raw_17['ÁREA GEOGRÁFICA'] == 'Total'].copy()

# Convertir a numérico y sumar 0-4
pob_17[cols_disponibles] = pob_17[cols_disponibles].apply(
    pd.to_numeric, errors='coerce'
)
pob_17['pob_0_4'] = pob_17[cols_disponibles].sum(axis=1)

# Estandarizar código DIVIPOLA a 5 dígitos
pob_17['cod_mun'] = pob_17['DPMP'].astype(str).str.zfill(5)

# Filtrar solo 2016 y 2017
pob_17_filtrado = (
    pob_17[pob_17['AÑO'].isin([2016, 2017])]
    [['cod_mun', 'AÑO', 'pob_0_4']]
    .rename(columns={'AÑO': 'año'})
    .reset_index(drop=True)
)

print(f"\n✓ Población 2016-2017 construida:")
print(f"  Filas:       {len(pob_17_filtrado)}")
print(f"  Municipios:  {pob_17_filtrado['cod_mun'].nunique()}")
print(f"  Años:        {sorted(pob_17_filtrado['año'].unique())}")
print(f"\n  Estadísticas pob_0_4:")
print(pob_17_filtrado['pob_0_4'].describe().round(0))


✓ Población 2016-2017 construida:
  Filas:       2244
  Municipios:  1039
  Años:        [np.int64(2016), np.int64(2017)]

  Estadísticas pob_0_4:
count      2244.0
mean       3348.0
std       16729.0
min          55.0
25%         516.0
50%        1099.0
75%        2398.0
max      478734.0
Name: pob_0_4, dtype: float64


**2. Calcular tasa x 1000 para sivigila 2016 - 2017**

In [46]:
# sivigila ya tiene los datos de 2016 y 2017 agregados por municipio-año
# Verificar que están
años_sivigila = sorted(sivigila['año'].unique())
print(f"Años en sivigila: {años_sivigila}")
assert 2016 in años_sivigila, "2016 no está en sivigila — verificar carga"
assert 2017 in años_sivigila, "2017 no está en sivigila — verificar carga"

# Filtrar solo 2016-2017 del sivigila agregado
siv_1617 = sivigila[sivigila['año'].isin([2016, 2017])].copy()

# Si sivigila ya tiene casos_totales por municipio-año, usar directamente
# Si no, agregar primero
if 'casos_totales' not in siv_1617.columns:
    siv_1617 = (
        siv_1617.groupby(['cod_mun', 'año'])
        .agg(casos_totales=('COD_EVE', 'count'))
        .reset_index()
    )

# Unir con población
siv_1617 = siv_1617.merge(pob_17_filtrado, on=['cod_mun', 'año'], how='left')

# Calcular tasa
siv_1617['tasa_x1000'] = (
    siv_1617['casos_totales'] / siv_1617['pob_0_4'] * 1000
).replace([np.inf, -np.inf], np.nan).round(2)

print(f"\n✓ Tasa calculada para 2016-2017:")
print(f"  Registros: {len(siv_1617)}")
print(f"  Nulos en tasa: {siv_1617['tasa_x1000'].isna().sum()}")
print(f"\n  Por año:")
print(siv_1617.groupby('año').agg(
    municipios=('cod_mun','count'),
    tasa_media=('tasa_x1000','mean'),
    nulos=('tasa_x1000', lambda x: x.isna().sum())
).round(2))

Años en sivigila: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

✓ Tasa calculada para 2016-2017:
  Registros: 1495
  Nulos en tasa: 1495

  Por año:
      municipios  tasa_media  nulos
año                                
2016         682         NaN    682
2017         813         NaN    813


In [47]:
df_lag = df_modelo[['cod_mun', 'año', 'tasa_x1000']].copy()

# Desplazar: el valor de año t pasa a ser el lag del año t+1
df_lag['año']     = df_lag['año'] + 1
df_lag = df_lag.rename(columns={'tasa_x1000': 'tasa_lag1'})

# Unir al panel principal
df_final = df_final.merge(df_lag, on=['cod_mun', 'año'], how='left')

# 2018 no tiene lag (no hay 2017) → se excluye del modelo
# El split ya empieza en 2019, así que está correcto

print(f"Nulos en tasa_lag1: {df_final['tasa_lag1'].isna().sum()}")
print(f"(Son los registros de 2018 que no tienen año anterior)")
print(f"\nCorrelación tasa_lag1 vs target:")
print(df_final[['tasa_lag1','tasa_x1000']].corr().round(3))

Nulos en tasa_lag1: 1534
(Son los registros de 2018 que no tienen año anterior)

Correlación tasa_lag1 vs target:
            tasa_lag1  tasa_x1000
tasa_lag1       1.000       0.633
tasa_x1000      0.633       1.000


**3. Base completa de tasas 2016 - 2017**

In [48]:
# Base histórica 2016-2017
base_1617 = siv_1617[['cod_mun', 'año', 'tasa_x1000']].copy()

# Base panel 2018-2025 (ya existente en df_modelo)
base_1824 = df_modelo[['cod_mun', 'año', 'tasa_x1000']].copy()

# Concatenar
base_tasas_completa = pd.concat(
    [base_1617, base_1824], ignore_index=True
).sort_values(['cod_mun', 'año'])

print(f"✓ Base completa de tasas 2016-2024:")
print(f"  Total registros: {len(base_tasas_completa):,}")
print(f"  Años disponibles: {sorted(base_tasas_completa['año'].unique())}")
print(f"  Municipios únicos: {base_tasas_completa['cod_mun'].nunique():,}")
print(f"\n  Cobertura por año:")
print(base_tasas_completa.groupby('año').agg(
    municipios=('cod_mun','count'),
    nulos_tasa=('tasa_x1000', lambda x: x.isna().sum())
))

✓ Base completa de tasas 2016-2024:
  Total registros: 7,819
  Años disponibles: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
  Municipios únicos: 1,141

  Cobertura por año:
      municipios  nulos_tasa
año                         
2016         682         682
2017         813         813
2018         862           0
2019         894           0
2020         829           0
2021         892           0
2022         911           0
2023         959           0
2024         977           0


**4. Construir lags y unir al df_final**

In [49]:
# Lag 1: tasa del año t-1
lag1 = base_tasas_completa.copy()
lag1['año'] = lag1['año'] + 1
lag1 = lag1.rename(columns={'tasa_x1000': 'tasa_lag1'})

# Lag 2: tasa del año t-2
lag2 = base_tasas_completa.copy()
lag2['año'] = lag2['año'] + 2
lag2 = lag2.rename(columns={'tasa_x1000': 'tasa_lag2'})

# Eliminar lags anteriores si ya existen en df_final
for col in ['tasa_lag1', 'tasa_lag2', 'tendencia',
            'ratio_cambio', 'media_movil_2a']:
    if col in df_final.columns:
        df_final = df_final.drop(columns=[col])

# Unir al panel principal
df_final = df_final.merge(
    lag1[['cod_mun', 'año', 'tasa_lag1']],
    on=['cod_mun', 'año'], how='left'
)
df_final = df_final.merge(
    lag2[['cod_mun', 'año', 'tasa_lag2']],
    on=['cod_mun', 'año'], how='left'
)

# Features derivados del lag
df_final['tendencia']      = df_final['tasa_lag1'] - df_final['tasa_lag2']
df_final['ratio_cambio']   = (
    df_final['tasa_lag1'] / (df_final['tasa_lag2'] + 1e-6)
).clip(0, 5)
df_final['media_movil_2a'] = (
    df_final['tasa_lag1'] + df_final['tasa_lag2']
) / 2

FEATURES_LAG = [
    'tasa_lag1', 'tasa_lag2',
    'tendencia', 'ratio_cambio', 'media_movil_2a'
]

# ── Diagnóstico de cobertura por año ──
print("=== COBERTURA DE LAGS CON DATOS REALES 2016-2017 ===")
print(f"\n  {'Año':>6} {'Total':>8} {'lag1 OK':>10} {'lag2 OK':>10} {'Completos':>12}")
print(f"  {'-'*50}")
for año in sorted(df_final['año'].unique()):
    df_a   = df_final[df_final['año'] == año]
    n_tot  = len(df_a)
    n_lag1 = df_a['tasa_lag1'].notna().sum()
    n_lag2 = df_a['tasa_lag2'].notna().sum()
    n_comp = df_a[FEATURES_LAG].notna().all(axis=1).sum()
    print(f"  {año:>6} {n_tot:>8} {n_lag1:>8} ({n_lag1/n_tot*100:.0f}%)"
          f" {n_lag2:>8} ({n_lag2/n_tot*100:.0f}%) {n_comp:>8} ({n_comp/n_tot*100:.0f}%)")

print(f"\n  Correlación tasa_lag1 vs tasa_x1000:")
print(df_final[['tasa_lag1','tasa_x1000']].corr().round(3))

=== COBERTURA DE LAGS CON DATOS REALES 2016-2017 ===

     Año    Total    lag1 OK    lag2 OK    Completos
  --------------------------------------------------
    2018      862        0 (0%)        0 (0%)        0 (0%)
    2019      894      773 (86%)        0 (0%)        0 (0%)
    2020      829      737 (89%)      724 (87%)      673 (81%)
    2021      892      742 (83%)      780 (87%)      674 (76%)
    2022      911      807 (89%)      750 (82%)      695 (76%)
    2023      959      844 (88%)      828 (86%)      765 (80%)
    2024      983      893 (91%)      852 (87%)      801 (81%)

  Correlación tasa_lag1 vs tasa_x1000:
            tasa_lag1  tasa_x1000
tasa_lag1       1.000       0.633
tasa_x1000      0.633       1.000


In [50]:
# VERIFICACIÓN ANTI-LEAKAGE

VARS_LEAKAGE = ['casos_totales', 'tasa_x1000', 'tasa_sintetica']
for v in VARS_LEAKAGE:
    assert v not in FEATURES_LAG, f"LEAKAGE: {v} en FEATURES_LAG"

# Verificar que lag1 de 2022 usa datos de 2021 (no de 2022)
ejemplo = df_final[
    (df_final['año'] == 2022) &
    (df_final['tasa_lag1'].notna())
].head(1)
cod_ej  = ejemplo['cod_mun'].values[0]
tasa_21 = base_tasas_completa[
    (base_tasas_completa['cod_mun'] == cod_ej) &
    (base_tasas_completa['año'] == 2021)
]['tasa_x1000'].values

if len(tasa_21) > 0:
    lag1_22 = ejemplo['tasa_lag1'].values[0]
    match   = abs(tasa_21[0] - lag1_22) < 0.01
    print(f"✓ Verificación lag1: municipio {cod_ej}")
    print(f"  tasa_x1000 en 2021: {tasa_21[0]:.2f}")
    print(f"  tasa_lag1 en 2022:  {lag1_22:.2f}")
    print(f"  Coinciden: {'✓ SÍ' if match else '✗ NO — revisar'}")

print(f"\n✓ Anti-leakage OK — lags usan solo información pasada")

✓ Verificación lag1: municipio 05001
  tasa_x1000 en 2021: 3.47
  tasa_lag1 en 2022:  3.47
  Coinciden: ✓ SÍ

✓ Anti-leakage OK — lags usan solo información pasada
